# Treino do Modelo de Grokking

Analisar os tipos das funções estimadas pelos neurônios em uma rede neural treinada.
O objetivo é fazer uma análise dinâmica conforme a rede neural é treinada até atingir o "Grokking" e identificar como essas funções modificam o espaço latente em termos de Homotopia e Isotopia.
Para analizar isso vamos utilizar o modelo original que observou o fenômeno de Grokking. Queremos entender as ativações de cada camada da rede neural em cada etapa do treino. A partir disso, vamos fazer uma análise harmônica para entender essas funções. Além de compreender o espaço de treino e o espaço de saída.

## DataSet

Como faremos operações binárias módulo algum primo, nosso espaço de treino é um subconjunto de $ \mathbb{Z}^3 = \{(x, y, z) | x, y, z \in \mathbb{Z}\} $. Assim, podemos entender o espaço latente também como um subespaço de $ \mathbb{Z}^3 $. Isso nos dá que nossa rede neural é uma função injetiva de $ \mathbb{Z}^3 \to \mathbb{Z}^3 $. Estamos considerando a base de dados original, sobre o primo $p = 97$.

In [1]:
!pip install "torch" "pytorch-lightning==1.0.0" "mod" "blobfile" "umap-learn" "numpy<2" "matplotlib"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48/48 [pytorch-lightning]ightning]nt]ver-cu12]2]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prs 1.0.9 requires click==8.1.8, but you have click 8.3.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from grok.data import ArithmeticTokenizer, ArithmeticDataset
import os

/tmp/jupyter-kernel.sODD/lib/python3.12/site-packages/pytorch_lightning/__init__.py:79: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [3]:
data_dir = "./data/"
os.makedirs(data_dir, exist_ok=True)

In [4]:
# tokens
tok = ArithmeticTokenizer(data_dir)
with open(os.path.join(data_dir, "tokens.txt"), "w") as f:
    f.write("\n".join(tok.itos))

In [5]:
# modular addition equations
eqs = ArithmeticDataset.make_data("+", shuffle=False)
with open(os.path.join(data_dir, "modular_addition.txt"), "w") as f:
    f.write("\n".join(eqs))
print(f"{len(eqs)} equações foram escritas para {data_dir}/addition_data.txt")

9409 equações foram escritas para ./data//addition_data.txt


# Coletando Ativações

In [6]:
import os, re, itertools
import torch
import umap
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import Callback
from pytorch_lightning import Trainer
from grok.training import TrainableTransformer, add_args
from grok.data import ArithmeticDataset, ArithmeticTokenizer

/tmp/jupyter-kernel.sODD/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
parser = add_args()
parser.set_defaults(logdir=os.environ.get("GROK_LOGDIR", "."))
hparams = parser.parse_args([]) 
hparams.datadir = os.path.abspath("data")
hparams.max_epochs = 10**6 + 1
hparams.max_steps = None
hparams.checkpoint_path = "./checkpoints/"
hparams.train_data_pct = 25
hparams.weight_decay = 0.1
hparams.anneal_lr = True
hparams.random_seed = 24
hparams.anneal_lr_steps = hparams.max_epochs
print(hparams)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TrainableTransformer(hparams).double().to(device)

tokenizer = model.train_dataset.tokenizer
train_ds, val_ds = model.train_dataset, model.val_dataset

target_layers = {
    "embedding": model.transformer.embedding,
    "decoder_0": model.transformer.decoder.blocks[0],
    "decoder_1": model.transformer.decoder.blocks[1],
    "linear": model.transformer.linear,
}

Namespace(random_seed=24, gpu=0, max_epochs=1000001, max_steps=None, batchsize=0, n_layers=2, n_heads=4, d_model=128, dropout=0.0, weight_noise=0.0, non_linearity='relu', max_context_len=50, math_operator='+', operand_length=None, train_data_pct=25, warmup_steps=10, anneal_lr_steps=1000001, anneal_lr=True, max_lr=0.001, weight_decay=0.1, weight_decay_kind='to_zero', noise_factor=0, save_activations=False, save_outputs=False, logdir='.', datadir='/home/gabriel.ribeiro/data', checkpoint_path='./checkpoints/')


In [8]:

class MetricsAndPredictionDumper(Callback):
    def __init__(self, tokenizer, train_ds, val_ds, target_layers, out_dir="predictions", batch_size=4096, save_every=100):
        self.tokenizer = tokenizer
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.target_layers = target_layers
        self.out_dir = out_dir
        self.batch_size = batch_size
        self.save_every = save_every  
        
        self.last_preds = None 

        os.makedirs(self.out_dir, exist_ok=True)
        
        # 1. Initialize Accuracy File
        self.acc_file = os.path.join(self.out_dir, "accuracy.csv")
        with open(self.acc_file, "w") as f:
            f.write("train_acc,test_acc\n")

        # 2. Auto-detect '=' token
        self.eq_token_id = self._find_token_id("=")
        print(f"--> Fast mode ready. '=' ID: {self.eq_token_id}")

        # 3. Pre-Calculate Ground Truth
        self.train_truth = self._extract_ground_truth(self.train_ds)
        self.test_truth = self._extract_ground_truth(self.val_ds)

        # 4. Generate Static Input Files
        self._generate_static_files()

    def _find_token_id(self, char):
        try:
            ids = self.tokenizer.encode(char)
            for i in ids:
                if self.tokenizer.decode(torch.tensor([i])).strip() == char:
                    return i
        except:
            pass
        return self.tokenizer.encode(char)[0]

    def _extract_ground_truth(self, dataset):
        truth = []
        for idx in range(len(dataset)):
            text = self.tokenizer.decode(dataset.data[idx])
            nums = [n for n in re.findall(r'\d+', text)]
            if len(nums) >= 3:
                truth.append(nums[2])
            else:
                truth.append("-999")
        return truth

    def _generate_static_files(self):
        def save_static(dataset, filename):
            path = os.path.join(self.out_dir, filename)
            if os.path.exists(path): return
            
            with open(path, "w") as f:
                f.write("operand_a,operand_b\n") # Header
                for idx in range(len(dataset)):
                    text = self.tokenizer.decode(dataset.data[idx])
                    nums = [int(n) for n in re.findall(r'\d+', text)]
                    if len(nums) >= 2:
                        f.write(f"{nums[0]}, {nums[1]}\n")
                    else:
                        f.write("-1, -1\n")
        
        save_static(self.train_ds, "static_train.txt")
        save_static(self.val_ds, "static_test.txt")

    def _predict_dataset(self, pl_module, dataset):
        predictions = []
        pl_module.eval()
        device = pl_module.device
        loader = DataLoader(dataset.data, batch_size=self.batch_size, shuffle=False)
        
        with torch.no_grad():
            for batch_ids in loader:
                batch_ids = batch_ids.to(device)
                logits, *_ = pl_module(batch_ids)
                
                eq_mask = (batch_ids == self.eq_token_id)
                if eq_mask.sum() == 0: 
                    target_indices = torch.zeros(batch_ids.size(0), dtype=torch.long, device=device)
                else:
                    target_indices = eq_mask.float().argmax(dim=1)

                row_indices = torch.arange(batch_ids.size(0), device=device)
                target_logits = logits[row_indices, target_indices, :]
                pred_ids = target_logits.argmax(dim=1).detach().cpu().tolist()
                
                for pid in pred_ids:
                    s = self.tokenizer.decode(torch.tensor([pid])).strip()
                    if not s: s = "SPACE"
                    elif s == "<|eos|>": s = "-1"
                    predictions.append(s)
        
        pl_module.train()
        return predictions

    def _get_activations(self, pl_module, dataset):
        """Helper to extract raw activations for a given dataset."""
        device = pl_module.device
        loader = DataLoader(dataset.data, batch_size=self.batch_size, shuffle=False)
        raw_activations = {name: [] for name in self.target_layers}
        batch_acts_storage = {}

        def get_hook(name):
            def hook(model, input, output):
                if isinstance(output, tuple):
                    act = output[0]
                else:
                    act = output
                batch_acts_storage[name] = act.detach()
            return hook

        handles = []
        for name, layer in self.target_layers.items():
            handles.append(layer.register_forward_hook(get_hook(name)))

        try:
            with torch.no_grad():
                for batch_ids in loader:
                    batch_ids = batch_ids.to(device)
                    batch_acts_storage = {}
                    
                    pl_module(batch_ids)

                    eq_mask = (batch_ids == self.eq_token_id)
                    eq_indices = eq_mask.float().argmax(dim=1)
                    
                    for name in self.target_layers:
                        if name not in batch_acts_storage: continue
                        layer_out = batch_acts_storage[name]
                        vecs = layer_out[torch.arange(layer_out.size(0)), eq_indices]
                        raw_activations[name].append(vecs.cpu().numpy())
        finally:
            for h in handles: h.remove()
            
        return raw_activations

    '''
    def _extract_activations_and_umap(self, pl_module, epoch, epoch_dir):
        print(f"--> Extracting train/test activations for epoch {epoch}...")
        pl_module.eval()

        # Extract for both datasets
        test_raw = self._get_activations(pl_module, self.val_ds)
        train_raw = self._get_activations(pl_module, self.train_ds)

        # UMAP and Save
        for name in self.target_layers:
            if not test_raw[name] or not train_raw[name]: continue

            test_full = np.concatenate(test_raw[name], axis=0) 
            train_full = np.concatenate(train_raw[name], axis=0)
            
            if np.any(np.isnan(test_full)) or np.any(np.isnan(train_full)):
                print(f"WARNING: NaN detected in {name}, skipping.")
                continue

            # Apply UMAP independently (can be combined if you want shared manifold space)
            reducer_test = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1)
            test_emb = reducer_test.fit_transform(test_full)
            
            reducer_train = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1)
            train_emb = reducer_train.fit_transform(train_full)
            
            # Save
            filename = f"{name}_umap.csv"
            filepath = os.path.join(epoch_dir, filename)
            
            with open(filepath, "w") as f:
                f.write("test_x,test_y,test_z,train_x,train_y,train_z\n")
                
                # zip_longest fills missing rows with None when one dataset runs out
                for t_row, tr_row in itertools.zip_longest(test_emb, train_emb, fillvalue=None):
                    
                    if t_row is not None:
                        t_str = f"{t_row[0]:.4f},{t_row[1]:.4f},{t_row[2]:.4f}"
                    else:
                        t_str = ",,"  # Empty columns for missing test rows
                        
                    if tr_row is not None:
                        tr_str = f"{tr_row[0]:.4f},{tr_row[1]:.4f},{tr_row[2]:.4f}"
                    else:
                        tr_str = ",," # Empty columns for missing train rows
                        
                    f.write(f"{t_str},{tr_str}\n")
        
        pl_module.train()
    '''

    # Não vamos aplicar mais o UMAP
    def _extract_and_save_activations(self, pl_module, epoch, epoch_dir):
        print(f"--> Extracting raw train/test activations for epoch {epoch}...")
        pl_module.eval()

        # Extract for both datasets
        test_raw = self._get_activations(pl_module, self.val_ds)
        train_raw = self._get_activations(pl_module, self.train_ds)

        for name in self.target_layers:
            if not test_raw[name] or not train_raw[name]: continue

            test_full = np.concatenate(test_raw[name], axis=0) 
            train_full = np.concatenate(train_raw[name], axis=0)
            
            if np.any(np.isnan(test_full)) or np.any(np.isnan(train_full)):
                print(f"WARNING: NaN detected in {name}, skipping.")
                continue

            # Dynamically determine dimensionality (D)
            D = test_full.shape[1]
            test_headers = [f"test_x{i+1}" for i in range(D)]
            train_headers = [f"train_x{i+1}" for i in range(D)]
            
            # 1. Pad the arrays so they are the same length (replaces zip_longest)
            max_len = max(len(test_full), len(train_full))
            
            if len(test_full) < max_len:
                pad_test = np.full((max_len - len(test_full), D), np.nan)
                test_full = np.vstack([test_full, pad_test])
                
            if len(train_full) < max_len:
                pad_train = np.full((max_len - len(train_full), D), np.nan)
                train_full = np.vstack([train_full, pad_train])

            # 2. Combine into one matrix and dump via Pandas
            combined_data = np.hstack([test_full, train_full])
            df = pd.DataFrame(combined_data, columns=test_headers + train_headers)
            
            filename = f"{name}_activations.csv"
            filepath = os.path.join(epoch_dir, filename)
            
            # Writes the entire block at once, vastly faster than a Python loop
            df.to_csv(filepath, index=False, float_format="%.6f")

        pl_module.train()
    
    def on_train_epoch_end(self, trainer, pl_module, *args, **kwargs):
        epoch = trainer.current_epoch
        
        should_save = (epoch % self.save_every == 0) or (epoch == trainer.max_epochs - 1)

        # 1. Calc Accuracy
        train_preds = self._predict_dataset(pl_module, self.train_ds)
        train_correct = sum(1 for p, t in zip(train_preds, self.train_truth) if p == t)
        train_acc = train_correct / len(self.train_ds) 
        
        test_preds = self._predict_dataset(pl_module, self.val_ds)
        test_correct = sum(1 for p, t in zip(test_preds, self.test_truth) if p == t)
        test_acc = test_correct / len(self.val_ds) 

        with open(self.acc_file, "a") as f:
            f.write(f"{train_acc * 100:.5f},{test_acc * 100:.5f}\n")

        # 2. Save Preds & UMAP or not
        if should_save:
            epoch_dir = os.path.join(self.out_dir, f"epoch_{epoch}")
            os.makedirs(epoch_dir, exist_ok=True)

            pred_file = os.path.join(epoch_dir, "predictions.txt")
            with open(pred_file, "w") as f:
                for z in test_preds:
                    f.write(f"{z}\n")
            
            # self._extract_activations_and_umap(pl_module, epoch, epoch_dir)
            self._extract_and_save_activations(pl_module, epoch, epoch_dir)


In [ ]:
from torch.utils.data import DataLoader

# --- RUNNER ---
callback = MetricsAndPredictionDumper(
    tokenizer, 
    train_ds, 
    val_ds, 
    target_layers,
    out_dir=f"raw-predictions-{hparams.train_data_pct}pct", 
    batch_size=4096,
    save_every=5000 
)

trainer = Trainer(
    max_epochs=hparams.max_epochs,
    max_steps=hparams.max_steps,
    callbacks=[callback],
    logger=False,
    gpus=1
)

print("Starting training...")

trainer.fit(model)

GPU available: True, used: True
INFO:lightning:GPU available: True, used: True
TPU available: False, using: 0 TPU cores
INFO:lightning:TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-462f829e-3e2c-5f7c-bc5e-04aeafd4d4a0]
INFO:lightning:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-462f829e-3e2c-5f7c-bc5e-04aeafd4d4a0]


--> Fast mode ready. '=' ID: 1
Starting training...


Set SLURM handle signals.
INFO:lightning:Set SLURM handle signals.

  | Name        | Type        | Params
--------------------------------------------
0 | transformer | Transformer | 455 K 
INFO:lightning:
  | Name        | Type        | Params
--------------------------------------------
0 | transformer | Transformer | 455 K 


/tmp/jupyter-kernel.sODD/lib/python3.12/site-packages/pytorch_lightning/utilities/distributed.py:45: UserWarning: The validation_epoch_end should not return anything as of 9.1.to log, use self.log(...) or self.write(...) directly in the LightningModule
  warnings.warn(*args, **kwargs)


Epoch 0: 100%|██████████| 6/6 [00:00<00:00, 26.86it/s, loss=5.198]
                              --> Extracting raw train/test activations for epoch 0...
Epoch 505: 100%|██████████| 6/6 [00:00<00:00, 118.95it/s, loss=0.000]
                              